In [1]:
import numpy as np

# Практические задания

#### 1. Проверка формул. Численно проверьте все 10 формул из таблиц на случайных x и A. Все ошибки должны быть < 1e-5.

##### Общая процедура проверки одной скалярной формулы по вектору

In [6]:
def check_gradient_vector(f, grad_ana, x, A, h0, tol):
    # Преобразуем x в массив NumPy
    x = np.asarray(x, dtype=float)

    max_error = 0.0
    all_passed = True

    # Перебираем координаты x
    for k in range(len(x)):
        # Масштабируем шаг для координат с большим модулем
        h = h0 * max(1.0, abs(x[k]))

        # Создаём копии, чтобы не менять исходный x
        x_plus = x.copy()
        x_minus = x.copy()

        # Сдвигаем только k-ю координату на +h и -h
        x_plus[k] = x_plus[k] + h
        x_minus[k] = x_minus[k] - h

        # Значения функции в двух соседних точках
        f_plus = f(x_plus, A)
        f_minus = f(x_minus, A)

        # Центральный градиент
        num = (f_plus - f_minus) / (2.0 * h)

        # Аналитический градиент
        ana = grad_ana(x, A)[k]

        # Абсолютная ошибка между численной и аналитической оценками
        abs_err = abs(num - ana)

        # Относительная ошибка
        rel_err = abs_err / max(1e-12, abs(ana))

        # Проверка с комбинированным допуском
        combined_ok = (abs_err <= tol + tol * abs(ana))

        if not combined_ok:
            all_passed = False

        # Выводим результаты для текущей координаты
        print(
            f"Координата {k + 1}: "
            f"num={num:.8g}, "
            f"ana={ana:.8g}, "
            f"abs_err={abs_err:.3e}, "
            f"rel_err={rel_err:.3e}"
        )

        # Запоминаем наибольшую абсолютную ошибку
        max_error = max(max_error, abs_err)

    return max_error, all_passed

# Делаем проверку
def f(x, A):
    return x @ A @ x

def grad_ana(x, A):
    return(A + A.T) @ x

A = np.array([[2.0, 1.0], [1.0, 3.0]])

x = np.array([0.5, -1.0])

max_error, passed = check_gradient_vector(f, grad_ana, x, A, h0=1e-5, tol=1e-6)

print(f"\nМаксимальная ошибка: {max_error}")
print(f"Все проверки пройдены: {passed}")

Координата 1: num=0, ana=0, abs_err=0.000e+00, rel_err=0.000e+00
Координата 2: num=-5, ana=-5, abs_err=1.165e-11, rel_err=2.331e-12

Максимальная ошибка: 1.1652900866465643e-11
Все проверки пройдены: True


##### Процедура проверки производной по матрице

In [7]:
def check_derivative_to_the_matrix(f, dA_ana, x, A, h0, tol):
    # Преобразуем входные данные в массивы NumPy
    x = np.asarray(x, dtype=float)
    A = np.asarray(A, dtype=float).copy()   # Копия A нужна, чтобы функция не изменила исходную матрицу

    max_error = 0.0
    all_passed = True

    # Перебираем индексы строк и столбцов
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):

            # Масштабируем шаг в зависимости от значения элемента A[i, j]
            h = h0 * max(1.0, abs(A[i, j]))

            # Создаём независимые копии матрицы для двух вычислений
            A_plus = A.copy()
            A_minus = A.copy()

            # Возмущаем только проверяемый элемент матрицы
            A_plus[i, j] += h
            A_minus[i, j] -= h

            # Значения функции при изменённом элементе матрицы
            f_plus = f(x, A_plus)
            f_minus = f(x, A_minus)

            # Численная производная по элементу A[i, j]
            num = (f_plus - f_minus) / (2.0 * h)

            # Вычисляем аналитическую матрицу производных
            ana = dA_ana(x, A)[i, j]

            abs_err = abs(num - ana)

            # Защита от деления на ноль при вычислении относительной ошибки
            rel_err = abs_err / max(1e-12, abs(ana))

            # Проверяем ошибку с абсолютной и относительной частями допуска
            combined_ok = (abs_err <= tol + tol * abs(ana))

            if not combined_ok:
                all_passed = False

            # Выводим результаты для текущей координаты
            print(
                f"A[{i + 1}, {j + 1}]: "
                f"num={num:.8g}, "
                f"ana={ana:.8g}, "
                f"abs_err={abs_err:.3e}, "
                f"rel_err={rel_err:.3e}"
            )

            # Запоминаем наибольшую абсолютную ошибку
            max_error = max(max_error, abs_err)

    return max_error, all_passed

# Делаем проверку
def f(x, A):
    return x @ A @ x

def dA_ana(x, A):
    return np.outer(x, x)

x = np.array([1.0, 2.0])

A = np.array([[3.0, 1.0], [1.0, 4.0]])

max_error, all_passed = check_derivative_to_the_matrix(f=f, dA_ana=dA_ana, x=x, A=A, h0=1e-5, tol=1e-6)

print(f"\nМаксимальная ошибка: {max_error}")
print(f"Все проверки пройдены: {all_passed}")

A[1, 1]: num=1, ana=1, abs_err=8.057e-11, rel_err=8.057e-11
A[1, 2]: num=2, ana=2, abs_err=7.572e-11, rel_err=3.786e-11
A[2, 1]: num=2, ana=2, abs_err=7.572e-11, rel_err=3.786e-11
A[2, 2]: num=4, ana=4, abs_err=2.620e-11, rel_err=6.551e-12

Максимальная ошибка: 8.056599831718358e-11
Все проверки пройдены: True


##### Проверка якобиана векторной функции

In [2]:
def jackobian_check(r, J_ana, x, h0=1e-5, tol=1e-6):
    # Преобразуем входные данные в массивы NumPy
    x = np.asarray(x, dtype=float)

    # Проверяем размерность выхода и делаем его одномерным массивом
    r_at_x = np.asarray(r(x), dtype=float).reshape(-1)

    m = len(r_at_x)   # число выходных компонент
    n = len(x)        # число входных переменных

    # Аналитический якобиан должен иметь размер m × n
    J_exact = np.asarray(J_ana(x), dtype=float)

    if J_exact.shape != (m, n):
        raise ValueError(
            f"J_ana(x) должен иметь форму {(m, n)}"
            f", но получена форма {J_exact.shape}"
        )

    J_num = np.zeros((m, n), dtype=float)
    max_error = 0.0
    all_passed = True

    # Каждая колонка якобиана соответствует одной координате x[j]
    for j in range(n):
        h = h0 * max(1.0, abs(x[j]))

        x_plus = x.copy()
        x_minus = x.copy()

        # Сдвигаем только j-ю координату входного вектора
        x_plus[j] += h
        x_minus[j] -= h

        r_plus = np.asarray(r(x_plus), dtype=float).reshape(-1)
        r_minus = np.asarray(r(x_minus), dtype=float).reshape(-1)

        if len(r_plus) != m or len(r_minus) != m:
            raise ValueError("Функция r должна возвращать вектор постоянной длины")

        # Вычисляем производную каждой выходной компоненты по x[j]
        for i in range(m):
            num = (r_plus[i] - r_minus[i]) / (2.0 * h)
            ana = J_exact[i, j]

            J_num[i, j] = num
            abs_err = abs(num - ana)
            max_error = max(max_error, abs_err)

            # Допуск включает абсолютную и относительную составляющие
            combined_ok = (abs_err <= tol + tol * abs(ana))

            if not combined_ok:
                all_passed = False

            print(
                f"J{i + 1}, {j + 1}: "
                f"численная={num:.8g}, "
                f"аналитическая={ana:.8g}, "
                f"ошибка={abs_err:.3e}, "
                f"проверка={'пройдена' if combined_ok else 'не пройдена'}"
            )

    return J_num, max_error, all_passed

# Делаем проверку
def r(x):
    x1, x2,  = x
    return np.array([x1**2 + x2, np.sin(x1 * x2)])

def J_ana(x):
    x1, x2 = x
    return np.array([[2 * x1, 1],
                     [x2 * np.cos(x1 * x2),
                      x1 * np.cos(x1 * x2)],
                      ])

x = np.array([1.2, 0.7])

J_num, max_error, all_passed = jackobian_check(r=r, J_ana=J_ana, x=x, h0=1e-5, tol=1e-6)

print("\nЧисленный якобиан:")
print(J_num)

print(f"\nМаксимальная ошибка: {max_error}")
print(f"Все проверки пройдены: {all_passed}")

J1, 1: численная=2.4, аналитическая=2.4, ошибка=2.350e-11, проверка=пройдена
J2, 1: численная=0.46722398, аналитическая=0.46722398, ошибка=3.472e-12, проверка=пройдена
J1, 2: численная=1, аналитическая=1, ошибка=1.565e-11, проверка=пройдена
J2, 2: численная=0.80095539, аналитическая=0.80095539, ошибка=2.023e-11, проверка=пройдена

Численный якобиан:
[[2.4        1.        ]
 [0.46722398 0.80095539]]

Максимальная ошибка: 2.3504753698944114e-11
Все проверки пройдены: True


##### Проверка регрессионного теста: якобиан, неквадратные размеры

In [3]:
def jackobian_check(r, J_ana, x, h0=1e-5, tol=1e-6):
    x = np.asarray(x, dtype=float)
    m = len(np.asarray(r(x)).reshape(-1))
    n = len(x)

    J_exact = np.asarray(J_ana(x), dtype=float)
    if J_exact.shape != (m, n):
        raise ValueError(
            f"Ожидалась форма якобиана {(m, n)}, "
            f"получена {J_exact.shape}"
        )

    J_num = np.zeros((m, n))
    max_error = 0.0
    all_passed = True

    # Внешний цикл — по входным переменным, то есть по столбцам.
    for j in range(n):
        h = h0 * max(1.0, abs(x[j]))

        x_plus = x.copy()
        x_minus = x.copy()
        x_plus[j] += h
        x_minus[j] -= h

        r_plus = np.asarray(r(x_plus), dtype=float).reshape(-1)
        r_minus = np.asarray(r(x_minus), dtype=float).reshape(-1)

        # Внутренний цикл — по выходным компонентам, то есть по строкам.
        for i in range(m):
            num = (r_plus[i] - r_minus[i]) / (2.0 * h)
            ana = J_exact[i, j]
            J_num[i, j] = num

            abs_err = abs(num - ana)
            max_error = max(max_error, abs_err)

            if abs_err > tol + tol * abs(ana):
                all_passed = False

    return J_num, max_error, all_passed

# Тест 1: R² → R³
def r_2_to_3(x):
    a, b = x
    return np.array([
        a + 2 * b,
        a * b,
        a**2 - b,
    ])

def J_2_to_3(x):
    a, b = x
    return np.array([
        [1,     2],
        [b,     a],
        [2 * a, -1],
    ])

x_2 = np.array([1.3, -0.7])
J_num_2, error_2, passed_2 = jackobian_check(
    r_2_to_3, J_2_to_3, x_2
)

assert J_num_2.shape == (3, 2)
assert passed_2, f"Тест R² → R³ не пройден: ошибка {error_2}"

print("R² → R³: пройден")
print("Форма:", J_num_2.shape)
print("Максимальная ошибка:", error_2)

# Тест 2: R³ → R²
def r_3_to_2(x):
    a, b, c = x
    return np.array([
        a * b + c**2,
        np.sin(a) + b * c,
    ])

def J_3_to_2(x):
    a, b, c = x
    return np.array([
        [b,          a, 2 * c],
        [np.cos(a),  c, b],
    ])

x_3 = np.array([0.8, -1.1, 0.6])
J_num_3, error_3, passed_3 = jackobian_check(
    r_3_to_2, J_3_to_2, x_3
)

assert J_num_3.shape == (2, 3)
assert passed_3, f"Тест R³ → R² не пройден: ошибка {error_3}"

print("\nR³ → R²: пройден")
print("Форма:", J_num_3.shape)
print("Максимальная ошибка:", error_3)

R² → R³: пройден
Форма: (3, 2)
Максимальная ошибка: 1.4983125851131263e-11

R³ → R²: пройден
Форма: (2, 3)
Максимальная ошибка: 1.7190471268691e-11


##### Главный цикл проверки всех формул

In [13]:
def check_formula_gradient(formula_name, func, grad_func, x, h=1e-5, tol=1e-5):
    # Аналитический градиент
    grad_analytic = grad_func(x)

    # Проверка размерности
    if grad_analytic.shape != x.shape:
        raise ValueError(
            f"Ошибка размерности: grad_analytic имеет форму {grad_analytic.shape}, "
            f"а должен иметь форму {x.shape}"
        )

    # Вычисляем численный градиент
    grad_numeric = np.zeros_like(x)
    for i in range(x.size):
        # Создаем вектор направления
        e = np.zeros_like(x)
        e.flat[i] = 1   # flat нужен для работы с многомерными массивами

        # Вычисляем значения функции в точках
        f_plus = func(x + h * e)
        f_minus = func(x - h * e)

        # Центральная разность
        grad_numeric.flat[i] = np.sum((f_plus - f_minus) / (2 * h))

    # Относительная ошибка
    tol_array = np.full_like(grad_analytic, tol)
    denom = np.maximum(
        np.maximum(np.abs(grad_analytic), np.abs(grad_numeric)), tol_array)
    error = np.abs(grad_numeric - grad_analytic) / denom
    max_error = np.max(error)

    if max_error < tol:
        print(f"[OK] Формула '{formula_name}' прошла проверку. Max error: {max_error:.2e}")
        return True
    else:
        # Диагностика: ищем, в каком месте ошибка
        problem_idx = np.argwhere(error > tol)
        print(f"[FAIL] Формула '{formula_name}' не прошла проверку. Max error: {max_error:.2e}")
        print(f"Проблемы в индексах: {problem_idx}")
        return False

# Тестовая функция и её производная
def f(x):
    return x ** 2

def df(x):
    return 2 * x

# Проверяем для вектора [1, 2, 3]
x_test = np.array([1.0, 2.0, 3.0])
check_formula_gradient("Квадрат", f, df, x_test, tol=1e-8)

[OK] Формула 'Квадрат' прошла проверку. Max error: 6.55e-12


True

#### 2. Якобиан softmax. Выведите якобиан softmax(z) по z. Подсказка: J_ij = s_i (δ_ij - s_j). Реализуйте на NumPy и сравните с численным.

In [7]:
def softmax(z: np.ndarray) -> np.ndarray:
    """
    Вычисляет softmax для вектора z.
    Для численной стабильности вычитаем максимум.
    """
    exp_z = np.exp(z - np.max(z))   # вычитаем max для стабильности
    return exp_z / np.sum(exp_z)

def jacobian_softmax(z: np.ndarray) -> np.ndarray:
    """
    Аналитический якобиан softmax: J_ij = s_i * (delta_ij - s_j).
    Возвращает матрицу размера (K, K).
    """
    s = softmax(z)
    K = len(s)
    # Создаём диагональную матрицу diag(s)
    diag_s = np.diag(s)
    # Внешнее произведение s * s^T даёт матрицу с элементами s_i * s_j
    outer_ss = np.outer(s, s)
    J = diag_s - outer_ss
    return J

def check_softmax_jacobian(K: int, seed: int=42):
    """
    Проверяет аналитический якобиан через численные конечные разности.
    """
    np.random.seed(seed)
    
    # Генерация случайного вектора z из нормального распределения
    z = np.random.normal(loc=0.0, scale=1.0, size=K)

    # Аналитический расчет
    J_analytic = jacobian_softmax(z)

    # Численный расчет (конечные разности)
    h = 1e-5
    J_numeric = np.zeros((K, K))

    for j in range(K):
        dz = np.zeros_like(z)
        dz[j] = h

        z_plus = z + dz
        z_minus = z - dz

        s_plus = softmax(z_plus)
        s_minus = softmax(z_minus)

        for i in range(K):
            derivative_approx = (s_plus[i] - s_minus[i]) / (2 * h)
            J_numeric[i, j] = derivative_approx

    # Сравнение
    error_matrix = np.abs(J_numeric - J_analytic)
    max_error = np.max(error_matrix)

    print(f"Размерность K = {K}")
    print(f"Максимальная ошибка между аналитическим и численным якобианом: {max_error:.3e}")

    # Симметрия: J должна быть симметричной (J == J.T)
    is_symmetric = np.allclose(J_analytic, J_analytic.T)
    
    # Нулевая сумма строк: sum(J[i,:]) должен быть равен 0
    row_sums = np.sum(J_analytic, axis=1)
    rows_sum_to_zero = np.allclose(row_sums, 0)

    print(f"Симметрия соблюдена: {is_symmetric}")
    print(f"Суммы строк равны нулю: {rows_sum_to_zero}")
    
    if max_error < 1e-5:
        print("Success")
    else:
        print("Fail")

# Пример запуска проверки
if __name__ == "__main__":
    check_softmax_jacobian(K=5, seed=42)

Размерность K = 5
Максимальная ошибка между аналитическим и численным якобианом: 2.866e-12
Симметрия соблюдена: True
Суммы строк равны нулю: True
Success


#### 3. Градиент линейной регрессии. Реализуйте grad_linreg(X, y, w). Обучите GD на синтетических данных. Сравните с решением через np.linalg.solve.

#### 4. Градиент logistic regression. Реализуйте grad_logreg(X, y, w) и обучите классификатор на двух классах из Iris. Точность должна быть >95%.

#### 5. Гессиан 2D-функции. Постройте контурный график f(x, y) = x² + 3y² − 2xy. Посчитайте гессиан. Является ли точка (0,0) минимумом?

#### 6. Метод Ньютона. Реализуйте: x ← x − H⁻¹ ∇f. Сравните скорость сходимости с обычным GD на квадратичной функции.

#### 7. Ridge regression. Выведите формулу для w в задаче ||Xw - y||² + λ||w||². Реализуйте и покажите, что при λ > 0 система решается даже при вырожденной X^T X.

#### 8. Backprop линейного слоя. Для слоя y = Wx + b выведите ∂L/∂W, ∂L/∂b, ∂L/∂x через известный ∂L/∂y. Сравните с тем, что делает PyTorch.